# NOTEBOOK 0: Project Overview & Research Plan

## 🌊 Satellite-Based Inland Water Quality Classification

---

**Project Type:** Research-Grade Machine Learning Study  
**Dataset:** AquaSat (Cleaned)  
**Task:** Multiclass Classification of Water Quality  
**Author:** Research ML Pipeline  
**Date:** February 2026

---

## 1. Problem Motivation

### 1.1 The Importance of Water Quality Monitoring

Inland water bodies—lakes, rivers, reservoirs, and estuaries—are critical resources for:
- **Drinking water supply** for billions of people worldwide
- **Agricultural irrigation** supporting food production
- **Ecosystem health** maintaining biodiversity
- **Recreational activities** impacting local economies
- **Industrial processes** requiring clean water inputs

Traditional water quality monitoring relies on **in-situ sampling**, which is:
- ❌ Expensive and labor-intensive
- ❌ Spatially limited (point measurements)
- ❌ Temporally sparse (weekly to monthly sampling)
- ❌ Difficult to scale across large regions

### 1.2 Satellite Remote Sensing as a Solution

Earth observation satellites (Landsat 5, 7, 8) provide:
- ✅ **Synoptic coverage** (entire water bodies in single images)
- ✅ **Regular revisit** (every 16 days per satellite)
- ✅ **Historical archive** (40+ years of data)
- ✅ **Cost-effective** monitoring at scale

### 1.3 The Machine Learning Opportunity

By combining satellite spectral data with in-situ water quality measurements, we can:
1. **Train predictive models** that estimate water quality from satellite observations
2. **Classify water quality categories** for rapid assessment
3. **Enable near real-time monitoring** across large spatial extents
4. **Support environmental decision-making** with data-driven insights

## 2. The AquaSat Dataset

### 2.1 Dataset Overview

**AquaSat** is a curated dataset matching satellite observations with in-situ water quality measurements across the United States. Our cleaned version contains:

| Characteristic | Value |
|---------------|-------|
| Total Records | ~57,000 |
| Time Period | 1984-2020 |
| Satellites | Landsat 5, 7, 8 |
| Water Body Types | Lakes, Streams, Estuaries |
| Geographic Coverage | Continental USA |

### 2.2 Key Features

**Spectral Bands:**
- `blue` - Blue band reflectance (440-520 nm)
- `green` - Green band reflectance (520-600 nm)
- `red` - Red band reflectance (630-690 nm)
- `nir` - Near-infrared reflectance (770-900 nm)
- `swir1` - Shortwave infrared 1 (1550-1750 nm)
- `swir2` - Shortwave infrared 2 (2080-2350 nm)

**Standard Deviations:**
- `blue_sd`, `green_sd`, `red_sd`, `nir_sd`, `swir1_sd`, `swir2_sd`

**Metadata:**
- `sat` - Satellite identifier (5, 7, 8)
- `path`, `row` - Landsat WRS-2 scene coordinates
- `pixelCount` - Number of pixels in extraction
- `clouds` - Cloud cover percentage
- `pwater` - Percentage water pixels

**Temporal Information:**
- `date_parsed` - Observation date
- `year`, `month` - Temporal components

**Spatial Information:**
- `lat`, `long` - Geographic coordinates
- `type` - Water body type (Lake, Stream, Estuary)

### 2.3 Target Variables (Water Quality Parameters)

| Parameter | Name | Unit | Significance |
|-----------|------|------|---------------|
| **chl_a** | Chlorophyll-a | μg/L | Indicator of algal biomass/eutrophication |
| **tss** | Total Suspended Solids | mg/L | Measure of water turbidity/sediment |
| **secchi** | Secchi Disk Depth | meters | Water transparency measure |

**Important Note:** We will use these parameters ONLY to **derive water quality classes**, not as direct prediction targets.

## 3. Why Multiclass Water Quality Classification?

### 3.1 Classification vs. Regression

**Regression Approach (NOT our choice):**
- Predict exact values of chl_a, tss, secchi
- Requires high precision in satellite calibration
- Errors compound across prediction ranges
- Less interpretable for decision-makers

**Classification Approach (Our choice):**
- Predict water quality categories (Good/Moderate/Poor)
- More robust to measurement uncertainty
- Directly actionable for management decisions
- Aligns with regulatory frameworks (EPA, EU Water Framework Directive)

### 3.2 Three-Class System Justification

| Class | Description | Management Implication |
|-------|-------------|------------------------|
| **Good** | Clear water, low algal content | Suitable for recreation, minimal intervention |
| **Moderate** | Some degradation, moderate turbidity | Monitoring recommended, potential warnings |
| **Poor** | High algae/sediment, low transparency | Intervention required, possible advisories |

### 3.3 Threshold Derivation Strategy

We will derive thresholds based on:
1. **Scientific literature** (EPA guidelines, WHO standards)
2. **Dataset distribution** (data-driven percentiles)
3. **Balance considerations** (avoiding extreme class imbalance)

Detailed threshold justification will be provided in **Notebook 02**.

## 4. Hybrid Modeling Approach

### 4.1 Model Architecture Overview

```
┌─────────────────────────────────────────────────────────────────────┐
│                        HYBRID MODEL ARCHITECTURE                     │
├─────────────────────────────────────────────────────────────────────┤
│                                                                      │
│   ┌─────────────────┐                                                │
│   │ Satellite Data  │                                                │
│   │ (Spectral Bands)│                                                │
│   └────────┬────────┘                                                │
│            │                                                         │
│            ▼                                                         │
│   ┌─────────────────┐      ┌──────────────────┐                      │
│   │ Random Forest / │ ───▶ │ Class Probabilities │                   │
│   │ XGBoost         │      │ (Good, Mod, Poor)   │                   │
│   └─────────────────┘      └──────────┬──────────┘                   │
│                                        │                             │
│   ┌─────────────────┐                  │                             │
│   │ Original Features│                 │                             │
│   └────────┬────────┘                  │                             │
│            │                           │                             │
│            └───────┬───────────────────┘                             │
│                    │                                                 │
│                    ▼                                                 │
│            ┌─────────────────┐                                       │
│            │ CONCATENATION   │                                       │
│            │ (Features + Probs)│                                     │
│            └────────┬────────┘                                       │
│                     │                                                │
│                     ▼                                                │
│            ┌─────────────────┐                                       │
│            │ Neural Network  │                                       │
│            │ (MLP Classifier)│                                       │
│            └────────┬────────┘                                       │
│                     │                                                │
│                     ▼                                                │
│            ┌─────────────────┐                                       │
│            │ Final Prediction│                                       │
│            │ (Water Quality) │                                       │
│            └─────────────────┘                                       │
│                                                                      │
└─────────────────────────────────────────────────────────────────────┘
```

### 4.2 Model Comparison Strategy

We will train and compare three approaches:

| Model | Description | Hypothesis |
|-------|-------------|------------|
| **Baseline (RF/XGBoost)** | Tree-based ensemble on raw features | Strong baseline, interpretable feature importance |
| **MLP** | Neural network on raw features | Can capture nonlinear interactions |
| **Hybrid** | MLP on features + RF probabilities | Combines tree ensemble knowledge with neural flexibility |

### 4.3 Evaluation Metrics

- **Accuracy**: Overall correct classification rate
- **Macro F1-Score**: Average F1 across classes (handles imbalance)
- **Confusion Matrix**: Detailed class-wise performance
- **Per-Class Precision/Recall**: Understanding class-specific errors

## 5. Research Add-Ons (Advanced Contributions)

Beyond core modeling, this project includes three advanced research contributions:

---

### 5.1 🌦️ Research Add-On 1: Seasonal & Climate Effects

**Motivation:**
Water quality varies significantly with seasons due to:
- Temperature-driven algal growth cycles
- Precipitation and runoff patterns
- Stratification dynamics in lakes

**Research Questions:**
1. How does predicted water quality vary across seasons?
2. Does model accuracy differ by season?
3. Are there seasonal biases in predictions?

**Methods (Notebook 06):**
- Extract temporal features (month, season)
- Analyze prediction distributions by season
- Evaluate model performance stratified by season
- Visualize seasonal trends and biases

---

### 5.2 📊 Research Add-On 2: Prediction Confidence & Uncertainty

**Motivation:**
Not all predictions are equally reliable. Understanding uncertainty:
- Helps identify cases requiring additional verification
- Supports risk-aware decision-making
- Enables selective deployment of monitoring resources

**Research Questions:**
1. How can we quantify prediction confidence?
2. Which predictions are most/least certain?
3. How does uncertainty correlate with prediction accuracy?

**Methods (Notebook 07):**
- Probabilistic outputs from classifiers
- Monte Carlo Dropout for neural uncertainty
- Confidence calibration analysis
- Low-confidence region identification

---

### 5.3 🔍 Research Add-On 3: Explainable Machine Learning

**Motivation:**
For environmental applications, understanding WHY a model makes predictions is crucial for:
- Scientific validation
- Trustworthy deployment
- Identifying potential model failure modes

**Research Questions:**
1. Which spectral features drive predictions?
2. How do features interact in decision-making?
3. Can we explain individual predictions?

**Methods (Notebook 08):**
- Random Forest feature importance
- SHAP (SHapley Additive exPlanations) analysis
- Feature interaction visualization
- Case study explanations

## 6. Evaluation Strategy

### 6.1 Data Splitting

```
Full Dataset (57,000 records)
          │
          ├──────────────────┬────────────────────┐
          │                  │                    │
          ▼                  ▼                    ▼
    ┌──────────┐      ┌──────────┐         ┌──────────┐
    │ TRAIN    │      │ VALIDATION│        │ TEST     │
    │ (60%)    │      │ (20%)     │        │ (20%)    │
    │ ~34,000  │      │ ~11,400   │        │ ~11,400  │
    └──────────┘      └──────────┘         └──────────┘
         │                  │                    │
         │                  │                    │
    Model Training    Hyperparameter      Final Evaluation
                      Tuning              (Never touched)
```

### 6.2 Stratified Sampling

All splits will be **stratified by water quality class** to ensure:
- Consistent class distributions across splits
- Fair evaluation of minority classes
- Reproducible results

### 6.3 Evaluation Protocol

1. **Train models** on training set only
2. **Tune hyperparameters** using validation set
3. **Final evaluation** on held-out test set
4. **Report all metrics** with confidence intervals where applicable

### 6.4 Data Leakage Prevention

Critical checks to prevent information leakage:
- ✅ Target variable (water_quality_class) derived BEFORE splitting
- ✅ Scaling/normalization fit on TRAINING data only
- ✅ Feature selection based on TRAINING data only
- ✅ No site-based or temporal overlap checks between splits

## 7. Project Structure

```
waterrr/
│
├── dataset_with_all_targets_present.csv    # Original cleaned dataset
│
├── notebooks/
│   ├── 00_Project_Overview_Research_Plan.ipynb      # This notebook
│   ├── 01_Data_Loading_Final_Cleaning.ipynb         # Data verification
│   ├── 02_Target_Engineering.ipynb                  # Water quality classes
│   ├── 03_Feature_Engineering_Preprocessing.ipynb   # Feature pipeline
│   ├── 04_Baseline_Multiclass_Model.ipynb           # RF/XGBoost
│   ├── 05_Neural_Network_Hybrid_Model.ipynb         # MLP & Hybrid
│   ├── 06_Climate_Seasonal_Effects.ipynb            # Research Add-On 1
│   ├── 07_Prediction_Confidence_Uncertainty.ipynb   # Research Add-On 2
│   ├── 08_Explainable_ML.ipynb                      # Research Add-On 3
│   └── 09_Final_Conclusions_Future_Work.ipynb       # Summary
│
├── processed_data/
│   ├── train.csv                           # Training split
│   ├── val.csv                             # Validation split
│   ├── test.csv                            # Test split
│   └── feature_scaler.pkl                  # Fitted scaler
│
├── models/
│   ├── baseline_model.pkl                  # RF/XGBoost model
│   ├── mlp_model.pt                        # PyTorch MLP
│   └── hybrid_model.pt                     # Hybrid model
│
└── figures/                                # Saved visualizations
```

## 8. Notebook Workflow Summary

| Notebook | Title | Purpose | Outputs |
|----------|-------|---------|--------|
| **00** | Project Overview | Define scope, methods, plan | This document |
| **01** | Data Loading | Load, verify, sanity check | Confirmed dataset |
| **02** | Target Engineering | Create water quality classes | `water_quality_class` column |
| **03** | Feature Engineering | Preprocess, split, scale | Train/Val/Test splits |
| **04** | Baseline Model | Train RF/XGBoost | Baseline accuracy, saved model |
| **05** | Neural Network & Hybrid | MLP and Hybrid models | Model comparison |
| **06** | Seasonal Effects | Analyze temporal patterns | Seasonal insights |
| **07** | Uncertainty Estimation | Quantify prediction confidence | Uncertainty metrics |
| **08** | Explainability | Feature importance, SHAP | Explainability report |
| **09** | Conclusions | Summarize findings | Final report |

## 9. Dependencies & Environment

### Python Packages Required:

```python
# Core
numpy
pandas
matplotlib
seaborn

# Machine Learning
scikit-learn
xgboost
imbalanced-learn

# Deep Learning
torch

# Explainability
shap

# Utilities
joblib
tqdm
```

### Installation:

```bash
pip install numpy pandas matplotlib seaborn scikit-learn xgboost imbalanced-learn torch shap joblib tqdm
```

## 10. Next Steps

Proceed to **Notebook 01: Data Loading & Final Cleaning** to:

1. ✅ Load the AquaSat dataset
2. ✅ Verify feature completeness
3. ✅ Confirm target availability
4. ✅ Perform final sanity checks
5. ✅ Generate summary statistics

---

**End of Notebook 00**